# Advanced Interactive Data Visualisation in Finance

This introduction sets expectations, justifies the focus on `go`, and smoothly transitions into code — perfect for a finance-oriented advanced Plotly notebook. This hands-on guide teaches readers on how to create powerful, publication-quality, and highly interactive financial visualizations using **Plotly** in Python.

While **Plotly Express** (`plotly.express`) offers an excellent high-level interface for rapid charting, many advanced financial use-cases demand finer control over trace behavior, layout composition, subplot synchronization, custom hover templates, annotations, shapes, secondary axes, candlestick/OHLC formatting, volume overlays, technical indicator layering, range selectors, and complex dashboard-style compositions. This is where **Plotly Graph Objects** (`import plotly.graph_objects as go`) becomes essential.

Let us dive into the precision and flexibility that `plotly.graph_objects` brings to financial storytelling.

In [ ]:
import plotly.graph_objects as go
import numpy as np
import pandas as pd
import yfinance as yf
import pandas_datareader.data as web
import warnings
import statsmodels.api as sm

from plotly.subplots import make_subplots
from datetime import datetime, timedelta

warnings.filterwarnings('ignore')

Download some stock data from YFinance

In [ ]:
# Tickers to download
tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA']
start_date = '2020-01-01'
end_date = '2024-01-01'

# Download historical data
data = yf.download(tickers, start=start_date, end=end_date)['Close']

#### 1) Basic Plotting with Plotly

In [ ]:
fig = go.Figure()
for ticker in tickers:
  fig.add_trace(go.Scatter(x=data.index, y=data[ticker], mode='lines', name=ticker))
fig.update_layout(title='Stock Prices Over Time', xaxis_title='Date', yaxis_title='Price (USD)')
fig.show()

We can stylise the above chart to make it more custom to our needs. For example, we can change the background colour, fonts and gridlines for ease of readability

In [ ]:
fig = go.Figure()
for ticker in tickers:
  fig.add_trace(
    go.Scatter(
      x=data.index,
      y=data[ticker],
      mode='lines',
      name=ticker,
      line=dict(width=2)          # slightly thicker for better visibility
    )
  )

fig.update_layout(
  title=dict(
    text='Stock Prices Over Time',
    font=dict(size=20),
    x=0.5,                      # center title
    xanchor='center'
  ),
  xaxis_title='Date',
  yaxis_title='Price (USD)',
  
  # ── Dark theme activation ──
  template='plotly_dark',
  
  # Explicit background control (optional but reinforces dark look)
  plot_bgcolor='rgba(0,0,0,0)',     # transparent plot area → grid shows nicely
  paper_bgcolor='rgba(0,0,0,0)',    # transparent outer area → matches notebook/jupyter dark mode
  
  # Improve readability on dark background
  font=dict(color='#e0e0e0'),           # light text
  legend=dict(
    bgcolor='rgba(40,40,50,0.6)',     # semi-transparent dark legend box
    bordercolor='#444',
    borderwidth=1,
    font=dict(size=12)
  ),
  hovermode='x unified',                # nice unified hover tooltip
  hoverlabel=dict(
    bgcolor='rgba(30,30,45,0.92)',
    font=dict(color='#f0f0f0')
  ),
  
  # Optional: nicer grid & zero lines
  xaxis=dict(
    gridcolor='rgba(120,120,120,0.18)',
    zerolinecolor='rgba(180,180,180,0.3)',
    showspikes=True,
    spikecolor='white',
    spikesnap='cursor'
  ),
  yaxis=dict(
    gridcolor='rgba(120,120,120,0.18)',
    zerolinecolor='rgba(180,180,180,0.3)'
  ),
  
  margin=dict(l=60, r=40, t=80, b=60)
)
fig.show()

#### 2) Animated Timeframes

Sampled on a weekly timeframe

In [ ]:
def plotly_animation(dataframe, tickers, title, n_frames, ani_duration):
  daily_data = dataframe.copy()  # keep original data intact
  tickers = list(daily_data.columns)
  n_points = len(daily_data)

  # Sub-sample frames if dataset is very large → smoother & faster animation
  indices = np.linspace(0, n_points-1, n_frames, dtype=int)

  # Prepare frames — each frame shows data up to index i for ALL tickers
  frames = [
    go.Frame(
      data=[
        go.Scatter(
          x=daily_data.index[:i],
          y=daily_data[ticker].iloc[:i],
          mode='lines',
          name=ticker,
          line=dict(width=2.2)
        )
        for ticker in tickers
      ],
      traces=list(range(len(tickers))),   # important: matches number of traces in initial figure
      name=f"step{k}"               # name must match slider step args
    )
    for k, i in enumerate(indices)
  ]

  # Initial (empty-ish) traces — must match number of traces in frames
  initial_traces = [
    go.Scatter(
      x=[daily_data.index[0]],
      y=[daily_data[ticker].iloc[0]],
      mode='lines',
      name=ticker,
      line=dict(width=2.2)
    )
    for ticker in tickers
  ]

  fig = go.Figure(
    data=initial_traces,
    layout=go.Layout(
      title=dict(
        text=title,
        x=0.5,
        font=dict(size=22)
      ),
      template="plotly_dark",               # or "plotly", "seaborn", etc.
      plot_bgcolor="rgba(0,0,0,0)",
      paper_bgcolor="rgba(0,0,0,0)",
      xaxis=dict(
        title="Date",
        range=[daily_data.index.min(), daily_data.index.max()],
        gridcolor="rgba(120,120,120,0.15)",
        zeroline=False,
        showspikes=True,
        spikemode="across",
        spikesnap="cursor",
        spikecolor="grey",
        spikedash="solid"
      ),
      yaxis=dict(
        title="Price (USD)",
        gridcolor="rgba(120,120,120,0.15)",
        zeroline=False,
        type="linear"   # change to "log" if desired
      ),
      hovermode="x unified",
      hoverlabel=dict(
        bgcolor="rgba(40, 40, 50, 0.85)",   # dark semi-transparent gray-black
        font=dict(
          size=13,
          color="#e0e0e0"                 # light gray/white for good readability
        ),
        bordercolor="rgba(100, 100, 120, 0.6)",  # subtle border (optional)
        align="left"
      ),
      legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
      ),
      margin=dict(l=70, r=40, t=90, b=120),
        
      # Play button (classic Quant/Trading style)
      updatemenus=[
        dict(
          type="buttons",
          direction="left",
          x=0.00,
          y=-0.04,
          xanchor="left",
          yanchor="top",
          showactive=False,
          buttons=[
            dict(
              label="▶ Play",
              method="animate",
              args=[
                None,
                dict(
                  frame=dict(duration=ani_duration, redraw=True), # Adjust speed here (ms per frame)
                  fromcurrent=True,
                  mode="immediate"
                )
              ]
            )
          ]
        )
      ],
      
      # Slider with date labels
      sliders=[
        dict(
          active=0,
          pad={"t": 45, "b": 10},
          x=0.00,
          y=-0.22,
          len=0.86,
          currentvalue={
            "prefix": "Date: ",
            "font": {"size": 14},
            "visible": True,
            "xanchor": "right", 
            "offset": -60
          },
          steps=[
            dict(
              method="animate",
              label=daily_data.index[i].strftime("%Y-%m-%d"),
              args=[
                [f"step{k}"],
                dict(
                  mode="immediate",
                  frame=dict(duration=0, redraw=True),
                  transition=dict(duration=0)
                )
              ]
            )
            for k, i in enumerate(indices)
          ]
        )
      ]
    ),
    frames=frames
  )
  fig.show(config={'displayModeBar': True})

In [ ]:
n_frames = 150
ani_duration = 50  # ms per frame
plotly_animation(data, tickers, "Stock Prices Over Time – Interactive Animation", n_frames, ani_duration)

#### 3) Regime Switching Model Visualisation

In [ ]:
rg_start_date = '2022-01-01'
rg_end_date = '2025-09-01'

# Fetch Fama-French Daily Factors
ff_factors = web.DataReader('F-F_Research_Data_Factors_Daily', 'famafrench', start=rg_start_date, end=rg_end_date)[0]
ff_factors = ff_factors / 100.0
ff_factors['Mkt_Cumulative'] = (1 + ff_factors['Mkt-RF']).cumprod()
dates_array = ff_factors.index

# Continuous Regime Detection and Shading
ff_factors['Sign'] = np.where(ff_factors['Mkt-RF'] >= 0, 1, -1)
ff_factors['Change'] = ff_factors['Sign'].diff().ne(0).cumsum()

shapes = []
# Robust regime block calculation without .apply()
regime_groups = ff_factors.groupby('Change')
regime_data = []

for _, group in regime_groups:
  regime_data.append({
    'start': group.index[0],
    'end': group.index[-1],
    'sign': group['Sign'].iloc[0]
  })
  
regimes = pd.DataFrame(regime_data)

# Close the weekend/holiday gaps
for i in range(len(regimes) - 1):
  regimes.loc[i, 'end'] = regimes.loc[i+1, 'start']

for _, row in regimes.iterrows():
  color = 'rgba(57, 255, 20, 0.12)' if row['sign'] == 1 else 'rgba(255, 51, 51, 0.12)'
  shapes.append(dict(
    type="rect", xref="x", yref="paper",
    x0=row['start'], x1=row['end'],
    y0=0, y1=1, fillcolor=color,
    layer="below", line_width=0,
  ))
  
indices = np.linspace(0, len(ff_factors)-1, 150, dtype=int)
frames = [go.Frame(
  data=[go.Scatter(
    x=dates_array[:i], 
    y=ff_factors['Mkt_Cumulative'].iloc[:i], 
    mode='lines', 
    line=dict(color='#14efff', width=3)
  )],
  name=f"step{i}"
) for i in indices]

fig = go.Figure(
  data=[go.Scatter(x=[dates_array[0]], y=[ff_factors['Mkt_Cumulative'].iloc[0]], mode='lines', line=dict(color="#14efff", width=3))],
  layout=go.Layout(
    title="<b>The Market Wind</b>: Fama-French Mkt-RF (Cumulative)",
    template="plotly_dark",
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    shapes=shapes,
    xaxis=dict(range=[dates_array.min(), dates_array.max() + pd.Timedelta(days=20)], gridcolor='rgba(128,128,128,0.1)', zeroline=False),
    yaxis=dict(title="Cumulative Excess Return", gridcolor='rgba(128,128,128,0.1)', zeroline=False),
    margin=dict(r=200, t=100, b=150, l=80),
    # Original Quant Guild Playbutton styling
    updatemenus=[dict(
      type="buttons", showactive=False, x=0.0, y=-0.12, xanchor="left", yanchor="top",
      buttons=[dict(label="▶ Play", method="animate", args=[None, dict(frame=dict(duration=30, redraw=True), fromcurrent=True, mode="immediate")])]
    )],
    sliders=[dict(
      active=0, pad={"t": 50, "b": 10}, x=0.12, len=0.88,
      currentvalue={"prefix": "Date: ", "font": {"color": "#14efff", "size": 14}},
      steps=[dict(method="animate", label=str(dates_array[i].date()), args=[[f"step{i}"], dict(mode="immediate", frame=dict(duration=0, redraw=True))]) for i in indices]
    )]
  ),
  frames=frames
)

# Structural Break Line
fig.add_vline(x='2025-01-01', line_width=2, line_dash="dash", line_color="red")
fig.show()

#### 4) Equity Strategy Visualisation (with sample strategy)

In [ ]:
# Sample ticker "AAPL"
tickers = ['AAPL']
ticker_start_date = '2020-01-01'
ticker_end_date = '2024-01-01'

daily_ticker_data = yf.download(tickers, start=ticker_start_date, end=ticker_end_date)['Close']
daily_ticker_data['daily_return'] = daily_ticker_data['AAPL'].pct_change()

n_steps = len(daily_ticker_data)

# Strategy simple MA crossover
daily_ticker_data['5_MA'] = daily_ticker_data['AAPL'].rolling(window=5).mean()
daily_ticker_data['9_MA'] = daily_ticker_data['AAPL'].rolling(window=9).mean()

take_profit_pct = 0.04  # 4% TP
stop_loss_pct = 0.02    # 2% SL

in_position = False
entry_price = 0.0
trade_pnls = []      
buy_indices = []  
sell_indices = []
equity_curve = [1.0]

# Simulate the strategy
for indx in range(n_steps):
  current_price = daily_ticker_data['AAPL'].iloc[indx]
  ma5 = daily_ticker_data['5_MA'].iloc[indx]
  ma9 = daily_ticker_data['9_MA'].iloc[indx]
  prev_ma5 = daily_ticker_data['5_MA'].iloc[indx-1] if indx > 0 else np.nan
  prev_ma9 = daily_ticker_data['9_MA'].iloc[indx-1] if indx > 0 else np.nan
  
  daily_strat_ret = 0.0
  
  if in_position:
    daily_strat_ret = daily_ticker_data['daily_return'].iloc[indx]
    unrealized_pnl = (current_price - entry_price) / entry_price
        
    # Exit Conditions
    if unrealized_pnl >= take_profit_pct or unrealized_pnl <= -stop_loss_pct or (ma5 < ma9 and prev_ma5 >= prev_ma9):
      in_position = False
      trade_pnls.append(unrealized_pnl)
      sell_indices.append(indx)
  else:
    # Entry Condition
    if ma5 > ma9 and prev_ma5 <= prev_ma9:
      in_position = True
      entry_price = current_price
      buy_indices.append(indx)

  equity_curve.append(equity_curve[-1] * (1 + daily_strat_ret))

equity_curve = equity_curve[1:]

# Calculate the performance metrics
wins = [p for p in trade_pnls if p > 0]
losses = [p for p in trade_pnls if p <= 0]

win_rate = len(wins) / len(trade_pnls) if trade_pnls else 0
avg_win = np.mean(wins) if wins else 0
avg_loss = np.mean(losses) if losses else 0

strat_series = pd.Series(equity_curve).pct_change().dropna()
sharpe_ratio = np.sqrt(252) * strat_series.mean() / strat_series.std() if strat_series.std() != 0 else 0

peak = pd.Series(equity_curve).cummax()
mdd = ((pd.Series(equity_curve) - peak) / peak).min()

strat_total_return = equity_curve[-1] - 1


# Animate the visual and display the metrics
frames = []
dates_array = daily_ticker_data.index
closes_array = daily_ticker_data['AAPL'].values
ma5_array = daily_ticker_data['5_MA'].values
ma9_array = daily_ticker_data['9_MA'].values
buy_idx_arr = np.array(buy_indices)
sell_idx_arr = np.array(sell_indices)
equity_array = np.array(equity_curve)

step_size = max(1, n_steps // 200) 
frame_indices = list(range(1, n_steps, step_size))
if frame_indices[-1] != n_steps:
  frame_indices.append(n_steps)

for k in frame_indices:
  t_x = dates_array[:k]
  
  # Top Chart (Row 1)
  tr_price = go.Scatter(x=t_x, y=closes_array[:k], mode='lines', line=dict(color='white', width=1.5), name='Close')
  tr_ma5 = go.Scatter(x=t_x, y=ma5_array[:k], mode='lines', line=dict(color='#00ffff', width=1.5), name='5 MA')
  tr_ma9 = go.Scatter(x=t_x, y=ma9_array[:k], mode='lines', line=dict(color='#ff00ff', width=1.5), name='9 MA')
  
  b_mask = buy_idx_arr < k
  s_mask = sell_idx_arr < k
  tr_buy = go.Scatter(x=dates_array[buy_idx_arr[b_mask]], y=closes_array[buy_idx_arr[b_mask]], mode='markers', marker=dict(color='#39ff14', size=8, symbol='triangle-up'), name='Buy')
  tr_sell = go.Scatter(x=dates_array[sell_idx_arr[s_mask]], y=closes_array[sell_idx_arr[s_mask]], mode='markers', marker=dict(color='#ff3333', size=8, symbol='triangle-down'), name='Sell')
  
  # Bottom Chart (Row 2): Strategy Equity Only
  tr_equity = go.Scatter(x=t_x, y=equity_array[:k], mode='lines', line=dict(color="#149dff", width=2), fill='tozeroy', fillcolor='rgba(20, 157, 255, 0.1)', name='Strategy')

  # Note: Passed 6 traces (5 top, 1 bottom) per frame
  frames.append(go.Frame(data=[tr_price, tr_ma5, tr_ma9, tr_buy, tr_sell, tr_equity], name=f"step{k}"))

# Initialise the figure
fig = make_subplots(
  rows=2, cols=1, 
  row_heights=[0.6, 0.4],
  subplot_titles=[
    "Price & Moving Average Crossover", 
    "Strategy Equity"
  ],
  vertical_spacing=0.15 
)

# Dummy traces to match the exact order of the 6 traces in our frames
for _ in range(5): 
  fig.add_trace(go.Scatter(x=[dates_array[0]], y=[closes_array[0]]), row=1, col=1)
fig.add_trace(go.Scatter(x=[dates_array[0]], y=[equity_array[0]]), row=2, col=1)
fig.frames = frames

# Format the layout and metrics
sliders = [dict(
  active=0, currentvalue={"prefix": "Step: "}, pad={"t": 0},
  x=0.15, len=0.85, y=-0.05,
  steps=[dict(
    method="animate",
    args=[[frames[idx].name], dict(mode="immediate", frame=dict(duration=0, redraw=True), transition=dict(duration=0))],
    label=str(k)
  ) for idx, k in enumerate(frame_indices)]
)]

metrics_text = (
  f"<b>Strategy Metrics</b><br><br>"
  f"Strat Return: {strat_total_return:.2%}<br><br>"
  f"Total Trades: {len(trade_pnls)}<br>"
  f"Win Prob: {win_rate:.1%}<br>"
  f"Loss Prob: {(1-win_rate):.1%}<br>"
  f"Avg Winner: {avg_win:.2%}<br>"
  f"Avg Loser: {avg_loss:.2%}<br><br>"
  f"Sharpe Ratio: {sharpe_ratio:.2f}<br>"
  f"Max Drawdown: {mdd:.2%}"
)

fig.add_annotation(
  text=metrics_text,
  xref="paper", yref="paper",
  x=1.25,          # ← just slightly outside 1, but usually still visible
  y=0.85,
  showarrow=False,
  align="left",
  font=dict(color="white", size=13),
  bgcolor="rgba(0,0,0,0.4)",          # ← slight background helps visibility
  bordercolor="#00ffff",
  borderwidth=1,
  borderpad=10
)

# Set plots and paper to completely transparent
fig.update_layout(
  height=700, width=1400,
  title_text="Quantitative Strategy: 5/9 MA Crossover with Hard TP/SL",
  plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
  showlegend=False, sliders=sliders, 
  margin=dict(t=80, b=100, r=300), 
  updatemenus=[{
    'type': 'buttons', 'x': 0.0, 'y': -0.1, 'xanchor': 'left', 'yanchor': 'top', 'direction': 'left', 'showactive': False,
    'buttons': [{
      'label': '▶ Play', 
      'method': 'animate', 
      'args': [None, {'frame': {'duration': 40, 'redraw': True}, 'transition': {'duration': 0}, 'fromcurrent': True, 'mode': 'immediate'}]
    }]
  }]
)

# Enforce static axes limits
min_price, max_price = closes_array.min() * 0.95, closes_array.max() * 1.05
min_eq = equity_array.min() * 0.95
max_eq = equity_array.max() * 1.05
fig.update_xaxes(title_text='Date', range=[dates_array[0], dates_array[-1]], row=1, col=1, gridcolor='rgba(128,128,128,0.2)')
fig.update_yaxes(title_text='Price', range=[min_price, max_price], row=1, col=1, gridcolor='rgba(128,128,128,0.2)')
fig.update_xaxes(title_text='Date', range=[dates_array[0], dates_array[-1]], row=2, col=1, gridcolor='rgba(128,128,128,0.2)')
fig.update_yaxes(title_text='Equity Multiplier', range=[min_eq, max_eq], row=2, col=1, gridcolor='rgba(128,128,128,0.2)')
fig.show()

#### 5) Active Strategy vs Benchmark Strategy



In [ ]:
tickers = ['AAPL']
benchmark_ticker = '^GSPC'
start_date = '2020-01-01'
end_date = '2025-09-01'

ticker_data = yf.download(tickers, start=start_date, end=end_date)['Close']
benchmark_data = yf.download(benchmark_ticker, start=start_date, end=end_date)['Close']

# ── Compute daily returns AFTER merging (safer) ─────────────────────────────
# Merge on date index (both should already be DatetimeIndex)
merged = pd.concat(
  [ticker_data, benchmark_data],
  axis=1
).dropna(how='any')  

# Add returns
merged['return_' + tickers[0]] = merged[tickers[0]].pct_change()
merged['return_' + benchmark_ticker]  = merged[benchmark_ticker].pct_change()
merged = merged.ffill()
merged_df = merged.dropna()   # remove first row with NaN returns

# Cutoff data before 2025-01-01
df = merged_df.copy()
df['Date'] = df.index
df = df[df['Date'] <= '2025-01-01'].reset_index(drop=True)

n_steps = len(df) 

# Calculate Moving Averages
df['5_MA'] = df[tickers[0]].rolling(window=5).mean()
df['9_MA'] = df[tickers[0]].rolling(window=9).mean()

take_profit_pct = 0.04  # 4% TP
stop_loss_pct = 0.02    # 2% SL

in_position = False
entry_price = 0.0
trade_pnls = []      
equity_curve = [1.0]

# Simulate the Strategy
for i in range(n_steps):
  current_price = df[tickers[0]].iloc[i]
  ma5 = df['5_MA'].iloc[i]
  ma9 = df['9_MA'].iloc[i]
  prev_ma5 = df['5_MA'].iloc[i-1] if i > 0 else np.nan
  prev_ma9 = df['9_MA'].iloc[i-1] if i > 0 else np.nan
  
  daily_strat_ret = 0.0
  
  if in_position:
    if 'Daily_Return' in df.columns:
      daily_strat_ret = df['Daily_Return'].iloc[i] 
    else:
      daily_strat_ret = (current_price - df[tickers[0]].iloc[i-1]) / df[tickers[0]].iloc[i-1]
        
    unrealized_pnl = (current_price - entry_price) / entry_price
    
    # Exit Conditions
    if unrealized_pnl >= take_profit_pct or unrealized_pnl <= -stop_loss_pct or (ma5 < ma9 and prev_ma5 >= prev_ma9):
      in_position = False
      trade_pnls.append(unrealized_pnl)
          
  else:
    # Entry Condition
    if ma5 > ma9 and prev_ma5 <= prev_ma9:
      in_position = True
      entry_price = current_price
          
  equity_curve.append(equity_curve[-1] * (1 + daily_strat_ret))

equity_curve = equity_curve[1:]

# Compute the benchmarks
# Asset & SPX Buy & Hold
asset_returns = df['return_' + tickers[0]].fillna(0).values if 'return_' + tickers[0] in df.columns else df[tickers[0]].pct_change().fillna(0).values
asset_equity = np.cumprod(1 + asset_returns)
spx_returns = df['return_' + benchmark_ticker].fillna(0).values
spx_equity = np.cumprod(1 + spx_returns)

strat_returns = pd.Series(equity_curve).pct_change().fillna(0).values

# Filter out days where strategy had 0 returns (flat/cash) for CAPM
active_mask = strat_returns != 0.0
spx_returns_active = spx_returns[active_mask]
strat_returns_active = strat_returns[active_mask]

# Calculate Beta and Alpha on active days
if np.var(spx_returns_active) > 0:
  X = sm.add_constant(spx_returns_active)
  model = sm.OLS(strat_returns_active, X).fit()
  alpha = model.params[0]
  beta = model.params[1]
else:
  alpha, beta = 0, 0

wins = [p for p in trade_pnls if p > 0]
losses = [p for p in trade_pnls if p <= 0]

win_rate = len(wins) / len(trade_pnls) if trade_pnls else 0
avg_win = np.mean(wins) if wins else 0
avg_loss = np.mean(losses) if losses else 0

strat_series = pd.Series(equity_curve).pct_change().dropna()
sharpe_ratio = np.sqrt(252) * strat_series.mean() / strat_series.std() if strat_series.std() != 0 else 0

peak = pd.Series(equity_curve).cummax()
mdd = ((pd.Series(equity_curve) - peak) / peak).min()

strat_total_return = equity_curve[-1] - 1
asset_total_return = asset_equity[-1] - 1
spx_total_return = spx_equity[-1] - 1

# Animation plot
frames = []

dates_array = df['Date'].values
equity_array = np.array(equity_curve)

step_size = max(1, n_steps // 200) 
frame_indices = list(range(1, n_steps, step_size))
if frame_indices[-1] != n_steps:
  frame_indices.append(n_steps)

# Pre-calculate regression line points for the scatterplot
if len(spx_returns_active) > 0:
  x_range = np.array([spx_returns_active.min(), spx_returns_active.max()])
else:
  x_range = np.array([-0.05, 0.05])
y_range = alpha + beta * x_range

for k in frame_indices:
  t_x = dates_array[:k]
  
  # Top Chart (Row 1): Equity Curves
  tr_asset_eq = go.Scatter(x=t_x, y=asset_equity[:k], mode='lines', line=dict(color='#00ffff', width=1.5), name='Asset B&H')
  tr_spx_eq = go.Scatter(x=t_x, y=spx_equity[:k], mode='lines', line=dict(color='#aaaaaa', width=1.5, dash='dot'), name='SPX B&H')
  tr_strat_eq = go.Scatter(x=t_x, y=equity_array[:k], mode='lines', line=dict(color='#39ff14', width=2), fill='tozeroy', fillcolor='rgba(57, 255, 20, 0.1)', name='Strategy')
  
  # Bottom Chart (Row 2): Scatterplot (Masked to remove flat days)
  t_strat_ret = strat_returns[:k]
  t_spx_ret = spx_returns[:k]
  t_mask = t_strat_ret != 0.0
  
  tr_scatter = go.Scatter(x=t_spx_ret[t_mask], y=t_strat_ret[t_mask], mode='markers', marker=dict(color='rgba(0, 255, 255, 0.5)', size=5), name='Active Returns')
  tr_beta = go.Scatter(x=x_range, y=y_range, mode='lines', line=dict(color='#ff00ff', width=2), name=f'Beta: {beta:.2f}')

  # Pass exactly 5 traces per frame
  frames.append(go.Frame(data=[tr_asset_eq, tr_spx_eq, tr_strat_eq, tr_scatter, tr_beta], name=f"step{k}"))

# Figure initialization with dummy traces to match the 5 traces in our frames
fig = make_subplots(
  rows=2, cols=1, 
  row_heights=[0.6, 0.4],
  subplot_titles=[
    "Strategy Equity vs. Benchmarks (Asset & SPX)", 
    f"Active Return Correlation (Beta: {beta:.2f})"
  ],
  vertical_spacing=0.15 
)

# Dummy traces matching the 5 traces in our frames
fig.add_trace(go.Scatter(x=[dates_array[0]], y=[asset_equity[0]]), row=1, col=1)
fig.add_trace(go.Scatter(x=[dates_array[0]], y=[spx_equity[0]]), row=1, col=1)
fig.add_trace(go.Scatter(x=[dates_array[0]], y=[equity_array[0]]), row=1, col=1)
fig.add_trace(go.Scatter(x=[0], y=[0]), row=2, col=1)
fig.add_trace(go.Scatter(x=x_range, y=y_range), row=2, col=1)
fig.frames = frames

# Layout and metrics formatting
sliders = [dict(
  active=0, currentvalue={"prefix": "Step: "}, pad={"t": 0},
  x=0.15, len=0.85, y=-0.05,
  steps=[dict(
    method="animate",
    args=[[frames[idx].name], dict(mode="immediate", frame=dict(duration=0, redraw=True), transition=dict(duration=0))],
    label=str(k)
  ) for idx, k in enumerate(frame_indices)]
)]

metrics_text = (
  f"<b>Performance Comparison</b><br><br>"
  f"Asset B&H: <span style='color:#00ffff'>{asset_total_return:.2%}</span><br>"
  f"SPX B&H: <span style='color:#aaaaaa'>{spx_total_return:.2%}</span><br>"
  f"Strategy: <span style='color:#39ff14'>{strat_total_return:.2%}</span><br><br>"
  f"<b>Strategy Metrics</b><br>"
  f"Total Trades: {len(trade_pnls)}<br>"
  f"Win Prob: {win_rate:.1%}<br>"
  f"Avg Winner: {avg_win:.2%}<br>"
  f"Avg Loser: {avg_loss:.2%}<br>"
  f"Sharpe Ratio: {sharpe_ratio:.2f}<br>"
  f"Max Drawdown: {mdd:.2%}<br><br>"
  f"<b>CAPM (Active Days)</b><br>"
  f"Beta: {beta:.2f}"
)

fig.add_annotation(
  text=metrics_text, xref="paper", yref="paper", 
  x=1.25, y=0.85,  
  showarrow=False, align="left", font=dict(color="white", size=13),
  bgcolor="rgba(0,0,0,0)", bordercolor="#00ffff", borderwidth=1, borderpad=10
)

# Set plots and paper to completely transparent
fig.update_layout(
  height=700, width=1400,
  title_text="Quantitative Strategy vs Benchmark Performance",
  plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
  showlegend=False, sliders=sliders, 
  margin=dict(t=80, b=100, r=300), 
  updatemenus=[{
    'type': 'buttons', 'x': 0.0, 'y': -0.1, 'xanchor': 'left', 'yanchor': 'top', 'direction': 'left', 'showactive': False,
    'buttons': [{
      'label': '▶ Play', 
      'method': 'animate', 
      'args': [None, {'frame': {'duration': 40, 'redraw': True}, 'transition': {'duration': 0}, 'fromcurrent': True, 'mode': 'immediate'}]
    }]
  }]
)

# Enforce static axes limits
min_eq = min(equity_array.min(), spx_equity.min(), asset_equity.min()) * 0.95
max_eq = max(equity_array.max(), spx_equity.max(), asset_equity.max()) * 1.05

if len(strat_returns_active) > 0:
  min_ret, max_ret = strat_returns_active.min() * 1.1, strat_returns_active.max() * 1.1
  min_spx, max_spx = spx_returns_active.min() * 1.1, spx_returns_active.max() * 1.1
else:
  min_ret, max_ret = -0.05, 0.05
  min_spx, max_spx = -0.05, 0.05

fig.update_xaxes(
    row=1, col=1,
    type='date',
    range=[str(dates_array.min()), str(dates_array.max() + pd.Timedelta(days=30))],
    tickformat='%Y-%m-%d',
    tickangle=-45,
    nticks=12,
    autorange=False,
    fixedrange=True
)
fig.update_yaxes(title_text='Equity Multiplier', range=[min_eq, max_eq], row=1, col=1, gridcolor='rgba(128,128,128,0.2)')
fig.update_xaxes(title_text='SPX Daily Return', range=[min_spx, max_spx], tickformat='.1%', row=2, col=1, gridcolor='rgba(128,128,128,0.2)')
fig.update_yaxes(title_text='Strategy Daily Return', range=[min_ret, max_ret], tickformat='.1%', row=2, col=1, gridcolor='rgba(128,128,128,0.2)')
fig.show()

#### 6) Beta Decay for Active Strategy

In [ ]:
tickers = ['AAPL']
benchmark_ticker = '^GSPC'
start_date = '2020-01-01'
end_date   = '2025-09-01'

ticker_data = yf.download(tickers, start=start_date, end=end_date)['Close']
benchmark_data = yf.download(benchmark_ticker, start=start_date, end=end_date)['Close']

# ── Compute daily returns AFTER merging (safer) ─────────────────────────────
# Merge on date index (both should already be DatetimeIndex)
merged = pd.concat(
  [ticker_data, benchmark_data],
  axis=1
).dropna(how='any')  

# Add returns
merged['return_' + tickers[0]] = merged[tickers[0]].pct_change()
merged['return_' + benchmark_ticker]  = merged[benchmark_ticker].pct_change()
merged = merged.ffill()
merged_df = merged.dropna()   # remove first row with NaN returns

# HARD CUTOFF: April 8th, 2025
df = merged_df.copy()
df['Date'] = df.index
df = df[df['Date'] <= '2025-04-20'].reset_index(drop=True)

regime_cutoff_idx = df[df['Date'] >= '2025-01-01'].index[0]
regime_date = df['Date'].iloc[regime_cutoff_idx]

# Active Strategy: Simple MA Crossover on the stock
df['5_MA'] = df[tickers[0]].rolling(window=5).mean()
df['9_MA'] = df[tickers[0]].rolling(window=9).mean()

take_profit_pct, stop_loss_pct = 0.04, 0.02
in_position, entry_price = False, 0.0
equity_curve = [1.0]

for i in range(len(df)):
  curr_p = df[tickers[0]].iloc[i]
  ma5, ma9 = df['5_MA'].iloc[i], df['9_MA'].iloc[i]
  prev_ma5 = df['5_MA'].iloc[i-1] if i > 0 else np.nan
  prev_ma9 = df['9_MA'].iloc[i-1] if i > 0 else np.nan
  
  daily_ret = 0.0
  if in_position:
    daily_ret = df['return_' + tickers[0]].iloc[i]
    unrealized = (curr_p - entry_price) / entry_price
    if unrealized >= take_profit_pct or unrealized <= -stop_loss_pct or (ma5 < ma9 and prev_ma5 >= prev_ma9):
      in_position = False
  else:
    if ma5 > ma9 and prev_ma5 <= prev_ma9:
      in_position, entry_price = True, curr_p
          
  equity_curve.append(equity_curve[-1] * (1 + daily_ret))

df['Strat_Equity'] = equity_curve[1:]
df['Strat_Ret'] = df['Strat_Equity'].pct_change().fillna(0)
df['Asset_Equity'] = np.cumprod(1 + df['return_' + tickers[0]].fillna(0))
df['SPX_Equity'] = np.cumprod(1 + df['return_' + benchmark_ticker].fillna(0))

def get_regime_metrics(data):
  s_ret = (data['Strat_Equity'].iloc[-1] / data['Strat_Equity'].iloc[0]) - 1
  a_ret = (data['Asset_Equity'].iloc[-1] / data['Asset_Equity'].iloc[0]) - 1
  m_ret = (data['SPX_Equity'].iloc[-1] / data['SPX_Equity'].iloc[0]) - 1
  
  active = data[data['Strat_Ret'] != 0]
  if len(active) > 2:
    model = sm.OLS(active['Strat_Ret'], active['return_' + benchmark_ticker]).fit()
    beta = model.params.iloc[0]
  else:
    beta = 0
  return s_ret, a_ret, m_ret, beta

s1, a1, m1, b1 = get_regime_metrics(df.iloc[:regime_cutoff_idx])
s2, a2, m2, b2 = get_regime_metrics(df.iloc[regime_cutoff_idx:])

# Plot the animation
frames = []
dates_array = df['Date'].values
step_size = max(1, len(df) // 150)
frame_indices = list(range(1, len(df), step_size))
if frame_indices[-1] != len(df): frame_indices.append(len(df))

for k in frame_indices:
  t_x = dates_array[:k]
  tr_asset = go.Scatter(x=t_x, y=df['Asset_Equity'].iloc[:k], line=dict(color='#00ffff', width=1.5), name='Asset')
  tr_spx = go.Scatter(x=t_x, y=df['SPX_Equity'].iloc[:k], line=dict(color='#aaaaaa', width=1.5, dash='dot'), name='SPX')
  tr_strat = go.Scatter(x=t_x, y=df['Strat_Equity'].iloc[:k], line=dict(color='#39ff14', width=2), fill='tozeroy', name='Strat')
  
  curr_scat = df.iloc[:k]
  active_scat = curr_scat[curr_scat['Strat_Ret'] != 0]
  tr_scat = go.Scatter(x=active_scat['return_' + benchmark_ticker], y=active_scat['Strat_Ret'], mode='markers', marker=dict(color='rgba(0, 255, 255, 0.4)', size=4))
  
  current_b = b1 if k < regime_cutoff_idx else b2
  x_range = np.array([df['return_' + benchmark_ticker].min(), df['return_' + benchmark_ticker].max()])
  tr_beta = go.Scatter(x=x_range, y=current_b * x_range, mode='lines', line=dict(color='#ff00ff', width=2))
  frames.append(go.Frame(data=[tr_asset, tr_spx, tr_strat, tr_scat, tr_beta], name=f"step{k}"))

fig = make_subplots(rows=2, cols=1, row_heights=[0.6, 0.4], vertical_spacing=0.15, subplot_titles=["The Beta Reckoning: Equity Decay", "Unconditional Beta Correlation"])

fig.add_trace(go.Scatter(x=[dates_array[0]], y=[1]), row=1, col=1)
fig.add_trace(go.Scatter(x=[dates_array[0]], y=[1]), row=1, col=1)
fig.add_trace(go.Scatter(x=[dates_array[0]], y=[1]), row=1, col=1)
fig.add_trace(go.Scatter(x=[0], y=[0]), row=2, col=1)
fig.add_trace(go.Scatter(x=[0,1], y=[0,1]), row=2, col=1)
fig.add_vline(x=regime_date, line_width=2, line_dash="dash", line_color="red", row=1, col=1)
fig.frames = frames

sliders = [dict(
  active=0, pad={"t": 60, "b": 10}, x=0.1, len=0.9,
  currentvalue={"prefix": "Date: ", "font": {"color": "#39ff14"}},
  steps=[dict(method="animate", label=str(df['Date'].iloc[k-1].date()), args=[[f"step{k}"], dict(mode="immediate", frame=dict(duration=0, redraw=True))]) for k in frame_indices]
)]

# ADD PADDING TO X-AXIS: End range is 30 days after the final date
end_padding = dates_array[-1] + pd.Timedelta(days=30)

fig.update_layout(
  height=850, width=1400, template="plotly_dark", 
  paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
  margin=dict(r=300, t=100, b=150), showlegend=False,
  sliders=sliders,
  updatemenus=[dict(
    type="buttons", showactive=False, x=0.02, y=-0.08,
    buttons=[dict(label="▶ Play", method="animate", args=[None, dict(frame=dict(duration=30, redraw=True), fromcurrent=True)])]
  )]
)

metrics_text = (
  f"<b>REGIME 1 (PRE-2025)</b><br>"
  f"SPX Ret: {m1:.2%}<br>Asset Ret: {a1:.2%}<br>Strat Ret: {s1:.2%}<br><b>Beta: {b1:.2f}</b><br><br>"
  f"<b>REGIME 2 (CRASH TO APR 8)</b><br>"
  f"SPX Ret: {m2:.2%}<br>Asset Ret: {a2:.2%}<br>Strat Ret: {s2:.2%}<br><b>Beta: {b2:.2f}</b>"
)
fig.add_annotation(text=metrics_text, xref="paper", yref="paper", x=1.3, y=0.5, showarrow=False, align="left", bordercolor="red", borderwidth=1, borderpad=10)

# Static boundaries with Padding on the X-axis for the Top Chart
fig.update_xaxes(
  type='date',                    # ← This is the key line
  tickformat='%Y-%m-%d',          # yyyy-mm-dd format
  tickangle=-45,                  # better readability if crowded
  nticks=12,                      # reasonable number of ticks
  tickfont=dict(size=11),
  row=1, col=1
)
fig.update_yaxes(range=[0.6, df['Asset_Equity'].max()*1.1], row=1, col=1)

# Scatter boundaries
fig.update_xaxes(range=[-0.07, 0.07], row=2, col=1)
fig.update_yaxes(range=[-0.07, 0.07], row=2, col=1)
fig.show()

This concludes the advanced visualisation of systematic strategy and financial data using Plotly.